# 06 — Supervised ranker (walk 1 MVP)

LightGBM `LGBMRanker(lambdarank)` over PCA text + structured + macro + aux text
features. Walk-1 only (train 2002-2007, val 2008, test 2009). Outputs land in
`artifacts/ranker/walk-001/`.

**Spec:** `docs/superpowers/specs/2026-05-16-supervised-ranker-design.md`.

Per the `machine-learning` skill: Optuna study persisted, early stopping on val
NDCG@30, no tuning on test, model + HPs + diagnostics versioned together.


## A. Setup

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import optuna
from lightgbm import early_stopping
import matplotlib.pyplot as plt

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import (
    DEFAULT_RANKER_PARAMS,
    assemble_walk_features,
    build_ranker,
    compute_excess_return_buckets,
    evaluate_ranker,
    load_walk_pca,
    project_text_to_pca,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'
TEST_START,  TEST_END  = '2009-01-01', '2009-12-31'

PANEL_DIR = processed_dir() / 'training_panel'
EMBED_DIR = processed_dir() / 'finbert_stockday_embed'
OUT_DIR   = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_OPTUNA_TRIALS = 15
N_BUCKETS = 32
TOP_K = 30
RANDOM_STATE = 42

print(f'walk {WALK_ID}')
print(f'  train: {TRAIN_START} .. {TRAIN_END}')
print(f'  val:   {VAL_START} .. {VAL_END}')
print(f'  test:  {TEST_START} .. {TEST_END}')
print(f'out_dir: {OUT_DIR.relative_to(repo_root())}')

## B. Load PCA + assemble train/val/test feature matrices

Reads the per-year shards in the three windows, projects text vecs through the
walk-1 PCA (n_pca=79), and assembles (X, y_excess, group_sizes, meta) triples
via `assemble_walk_features`. Memory ceiling: ~470k train rows × ~196 cols ≈ 700 MB.

In [ ]:
pca, n_pca = load_walk_pca(WALK_ID)
print(f'PCA loaded: n_components = {n_pca}')


def _load_panel_years(start: str, end: str) -> pd.DataFrame:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p)
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _load_embed_years(start: str, end: str) -> pd.DataFrame:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((EMBED_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p, columns=['permno', 'date', 'vec'])
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _assemble(window_start: str, window_end: str, label: str):
    panel = _load_panel_years(window_start, window_end)
    embed = _load_embed_years(window_start, window_end)
    embed_pca = project_text_to_pca(embed, pca)
    X, y_excess, groups, meta = assemble_walk_features(panel, embed_pca)
    print(f'{label}: panel={len(panel):>7}  embed={len(embed):>7}  '
          f'X={X.shape}  groups={len(groups)} Fridays')
    return X, y_excess, groups, meta


X_train, y_train_excess, groups_train, meta_train = _assemble(TRAIN_START, TRAIN_END, 'train')
X_val,   y_val_excess,   groups_val,   meta_val   = _assemble(VAL_START,   VAL_END,   'val  ')
X_test,  y_test_excess,  groups_test,  meta_test  = _assemble(TEST_START,  TEST_END,  'test ')

# Lambdarank labels (32 buckets on per-date excess return).
buckets_train = compute_excess_return_buckets(meta_train, n_buckets=N_BUCKETS).astype(int).values
buckets_val   = compute_excess_return_buckets(meta_val,   n_buckets=N_BUCKETS).astype(int).values

print(f'feature columns: {X_train.shape[1]}')
print(f'first 8 feature cols: {list(X_train.columns[:8])}')
print(f'NaN-rich cols in train (top 5): '
      f'{X_train.isna().mean().sort_values(ascending=False).head(5).to_dict()}')

## C. Optuna HP search

15 TPE trials over `num_leaves`, `learning_rate`, `min_data_in_leaf`,
`feature_fraction`, `bagging_fraction`, `lambda_l2`. Objective: val NDCG@30.
MedianPruner kills slow-converging trials. Study persisted to
`artifacts/ranker/walk-001/optuna_study.pkl` for later inspection.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'lambda_l2':        trial.suggest_float('lambda_l2', 0.0, 5.0),
        'n_estimators':     500,
    }
    model = build_ranker(params)
    model.fit(
        X_train, buckets_train,
        group=groups_train,
        eval_set=[(X_val, buckets_val)],
        eval_group=[groups_val],
        eval_at=[TOP_K],
        callbacks=[early_stopping(stopping_rounds=30, verbose=False)],
    )
    return float(model.best_score_['valid_0'][f'ndcg@{TOP_K}'])


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'best NDCG@{TOP_K}: {study.best_value:.4f}')
print(f'best params: {json.dumps(study.best_params, indent=2)}')
joblib.dump(study, OUT_DIR / 'optuna_study.pkl')
print(f'study persisted -> {(OUT_DIR / "optuna_study.pkl").relative_to(repo_root())}')

## D. Final fit with best HPs + early stopping on val

In [ ]:
best_params = {**study.best_params, 'n_estimators': 2000}
model = build_ranker(best_params)
model.fit(
    X_train, buckets_train,
    group=groups_train,
    eval_set=[(X_val, buckets_val)],
    eval_group=[groups_val],
    eval_at=[TOP_K],
    callbacks=[early_stopping(stopping_rounds=50, verbose=True)],
)
best_iter = int(model.best_iteration_)
val_ndcg = float(model.best_score_['valid_0'][f'ndcg@{TOP_K}'])
print(f'best iteration: {best_iter}, val NDCG@{TOP_K}: {val_ndcg:.4f}')

## E. OOS evaluation + feature importance

Rank IC + decile spread + hit rate + top-K Jaccard on the held-out 2009 test
window. Sanity gate: `test rank IC mean > 0` (else abort — model isn't ranking).
Top-20 features by gain plotted inline.

In [ ]:
test_metrics = evaluate_ranker(model, X_test, y_test_excess, meta_test['date'], top_k=TOP_K)
print('test metrics:')
for k, v in test_metrics.items():
    print(f'  {k:>20}: {v:.4f}')

assert test_metrics['rank_ic_mean'] > 0, (
    f"sanity gate failed: test rank IC mean = {test_metrics['rank_ic_mean']:.4f} <= 0"
)

fi = pd.DataFrame({
    'feature': X_train.columns,
    'gain': model.booster_.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=False)
print('\ntop 20 features by gain:')
print(fi.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 7))
fi.head(20).iloc[::-1].plot.barh(x='feature', y='gain', ax=ax, legend=False)
ax.set_title(f'Top 20 features by gain (walk {WALK_ID})')
plt.tight_layout()
plt.show()

## F. Persist artifacts

Writes `model.joblib` (model + feature names), `hp.json`, `summary.json`, and
`feature_importance.csv`. The `optuna_study.pkl` was already persisted in cell C.
Walk-1 is now complete; downstream notebook 07 (RL) can consume `model.joblib`.

In [ ]:
joblib.dump(
    {'model': model, 'feature_names': X_train.columns.tolist()},
    OUT_DIR / 'model.joblib',
)
(OUT_DIR / 'hp.json').write_text(json.dumps({
    **study.best_params,
    f'val_ndcg_at_{TOP_K}': val_ndcg,
    'best_iteration': best_iter,
}, indent=2))
fi.to_csv(OUT_DIR / 'feature_importance.csv', index=False)

summary = {
    'walk_id': WALK_ID,
    'train_window': [TRAIN_START, TRAIN_END],
    'val_window':   [VAL_START,   VAL_END],
    'test_window':  [TEST_START,  TEST_END],
    'n_features': int(X_train.shape[1]),
    'n_pca': int(n_pca),
    'n_train_rows': int(len(X_train)),
    'n_val_rows':   int(len(X_val)),
    'n_test_rows':  int(len(X_test)),
    'best_iteration': best_iter,
    f'val_ndcg_at_{TOP_K}': val_ndcg,
    **{f'test_{k}': float(v) for k, v in test_metrics.items()},
    'passed_sanity': bool(test_metrics['rank_ic_mean'] > 0),
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))

print(f'wrote artifacts to {OUT_DIR.relative_to(repo_root())}/')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')
print('\nsummary:')
print(json.dumps(summary, indent=2))